# 03 - Actor, Critic, Distribution

本节目标: 你能手写一个 Gaussian actor, 知道 `sample`, `log_prob`, `entropy`, `KL` 为什么是 PPO 必需的。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(1)

In [ ]:
class TinyActor(nn.Module):
    def __init__(self, obs_dim: int, action_dim: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, 32), nn.ELU(), nn.Linear(32, action_dim))
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def forward(self, obs: torch.Tensor):
        mean = self.net(obs)
        std = self.log_std.exp().expand_as(mean)
        return torch.distributions.Normal(mean, std)

class TinyCritic(nn.Module):
    def __init__(self, obs_dim: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, 32), nn.ELU(), nn.Linear(32, 1))

    def forward(self, obs: torch.Tensor):
        return self.net(obs)

obs = torch.randn(5, 10)
actor = TinyActor(obs_dim=10, action_dim=3)
critic = TinyCritic(obs_dim=10)

dist = actor(obs)
actions = dist.sample()
log_prob = dist.log_prob(actions).sum(dim=-1)
entropy = dist.entropy().sum(dim=-1)
values = critic(obs)

print('actions:', actions.shape)
print('log_prob:', log_prob.shape)
print('entropy:', entropy.shape)
print('values:', values.shape)

## old policy vs new policy

PPO 的 ratio 是:

$$\frac{\pi_\theta(a|s)}{\pi_{old}(a|s)} = \exp(\log \pi_\theta(a|s) - \log \pi_{old}(a|s))$$

rollout 时存 old log_prob, update 时重新算 new log_prob。

In [ ]:
with torch.no_grad():
    old_dist = actor(obs)
    old_actions = old_dist.sample()
    old_log_prob = old_dist.log_prob(old_actions).sum(dim=-1)

# 假装 actor 已经是 update 时的当前策略
new_dist = actor(obs)
new_log_prob = new_dist.log_prob(old_actions).sum(dim=-1)
ratio = torch.exp(new_log_prob - old_log_prob)

print('old_log_prob:', old_log_prob)
print('new_log_prob:', new_log_prob)
print('ratio:', ratio)

## 作业

1. 为什么 `log_prob(actions).sum(dim=-1)` 要 sum?
2. 把 action_dim 改成 12, 观察 log_prob shape 有没有变。
3. 打印 `actor.log_std`, 然后解释它是不是网络参数。